# Chapter 7 — Batch API and Cost Optimization · Homework (self-test)

**Goal:** check that you can design a batch pipeline that survives failures, pick the right processing tier and caching layout, handle rate limits and errors correctly, and put a defensible number on cost.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 60–90 minutes
- Estimated API cost: **$0**. Everything runs offline against fixtures built from real `openai` SDK types.
- Prerequisites: the chapter README and the practice notebook

**How to use it**

1. Attempt everything **without** scrolling to the solutions at the bottom.
2. Part A: answer 11 scenario questions in the `answers` cell, then run the grader (the key is stored as hashes).
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they print ✓ or a hint.
4. Part C: write a short architecture answer, then compare it with the rubric.
5. Run the scorecard. **Pass mark: 72%** (quiz and exercises weighted equally), the same bar as the Claude path.

In [ ]:
import hashlib
import json
import os
import re
from datetime import datetime, timedelta, timezone
from typing import Callable

from dotenv import find_dotenv, load_dotenv
from openai.types.responses import Response, ResponseUsage
from openai.types.responses.response_create_params import ResponseCreateParamsNonStreaming
from pydantic import BaseModel, TypeAdapter, ValidationError

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
print("model used in request-building exercises:", MODEL)

In [ ]:
# This homework runs fully offline (fixtures built from real SDK types, no API calls, no cost).
# The key check is informational only, so a missing key never blocks the self-test.
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")), "(not needed for this homework)")

In [ ]:
# Shared fixtures used by the exercises and the checkers. Run this cell, do not edit it.
INSTRUCTIONS = (
    "# Identity\nYou extract invoice fields for accounts payable.\n\n"
    "# Instructions\n* Dates ISO 8601, currency ISO 4217. Use null for any value not in the document."
)
INVOICE_SCHEMA = {
    "type": "object",
    "properties": {"vendor": {"type": ["string", "null"]}, "invoice_date": {"type": ["string", "null"]},
                   "currency": {"type": ["string", "null"]}, "total": {"type": ["number", "null"]}},
    "required": ["vendor", "invoice_date", "currency", "total"],
    "additionalProperties": False,
}
BODY_KEYS = ResponseCreateParamsNonStreaming.__required_keys__ | ResponseCreateParamsNonStreaming.__optional_keys__
BODY_ADAPTER = TypeAdapter(ResponseCreateParamsNonStreaming)
CUSTOM_ID_RE = re.compile(r"^[A-Za-z0-9_-]{1,64}$")


class Invoice(BaseModel):          # every field required and nullable, like INVOICE_SCHEMA
    vendor: str | None
    invoice_date: str | None
    currency: str | None
    total: float | None


# USD per 1M tokens, Standard, short context (pricing page, 2026-10-03)
PRICES = {
    "gpt-6-luna":  {"input": 0.10,  "cached": 0.01, "write": 0.125, "output": 0.50},
    "gpt-6.1-sol": {"input": 2.00,  "cached": 0.10, "write": 2.50,  "output": 10.00},
    "gpt-6-astra": {"input": 10.00, "cached": 1.00, "write": 12.50, "output": 50.00},
}
TIER_MULTIPLIER = {"default": 1.0, "batch": 0.5, "flex": 0.5, "fast": 2.0, "priority": 2.0}
STOP_CODES = {"insufficient_quota", "credit_balance_exhausted", "organization_spend_limit_exceeded",
              "project_spend_limit_exceeded", "organization_usage_limit_exceeded"}


def usage_of(input_tokens, output_tokens, *, cached=0, written=0, reasoning=0) -> ResponseUsage:
    return ResponseUsage.model_validate({
        "input_tokens": input_tokens, "output_tokens": output_tokens, "total_tokens": input_tokens + output_tokens,
        "input_tokens_details": {"cached_tokens": cached, "cache_write_tokens": written},
        "output_tokens_details": {"reasoning_tokens": reasoning}})


def body_of(texts, *, status="completed", reason=None, extra_items=()) -> dict:
    """A raw /v1/responses body (as it appears in a batch output file), shape-checked by the SDK's Response type."""
    output = list(extra_items)
    if texts is not None:
        output.append({"type": "message", "id": "msg_1", "role": "assistant", "status": "completed",
                       "content": [{"type": "output_text", "text": t, "annotations": []} for t in texts]})
    data = {"id": "resp_1", "object": "response", "created_at": 0, "model": MODEL, "status": status,
            "incomplete_details": {"reason": reason} if reason else None, "output": output,
            "parallel_tool_calls": True, "tool_choice": "auto", "tools": [],
            "usage": usage_of(500, 120, reasoning=64).model_dump()}
    return Response.model_validate(data).model_dump(mode="json", exclude_none=True)


def ok_line(cid, code=200, body=None):
    return {"id": f"batch_req_{cid}", "custom_id": cid,
            "response": {"status_code": code, "request_id": "req_1", "body": body}, "error": None}


GOOD = json.dumps({"vendor": "Acme Ltd", "invoice_date": "2026-03-03", "currency": "EUR", "total": 120.0})
LINES = {   # custom_id -> (line, expected bucket)
    "a__v3": (ok_line("a__v3", body=body_of([GOOD])), "ok"),
    "b__v3": ({"id": "x", "custom_id": "b__v3", "response": None,
               "error": {"code": "batch_expired", "message": "This request could not be executed before the completion window expired."}}, "resubmit"),
    "c__v3": (ok_line("c__v3", 500, {"error": {"message": "server error", "type": "server_error", "param": None, "code": None}}), "resubmit"),
    "d__v3": (ok_line("d__v3", 429, {"error": {"message": "rate limited", "type": "rate_limit_error", "param": None, "code": None}}), "resubmit"),
    "e__v3": (ok_line("e__v3", 400, {"error": {"message": "context too long", "type": "invalid_request_error", "param": "input", "code": None}}), "fix_first"),
    "f__v3": (ok_line("f__v3", body=body_of(None, status="incomplete", reason="max_output_tokens")), "fix_first"),
    "g__v3": (ok_line("g__v3", body=body_of(["I can't help with that document."])), "review"),
    "h__v3": (ok_line("h__v3", body=body_of([GOOD.replace('"EUR"', '"€"')])), "review"),
    "i__v3": (ok_line("i__v3", body=body_of([GOOD.replace('"EUR"', "null")])), "ok"),
    "j__v3": (ok_line("j__v3", body=body_of([GOOD[:40], GOOD[40:]],
                                           extra_items=[{"type": "reasoning", "id": "rs_1", "summary": []}])), "ok"),
    "k__v3": ({"id": "x", "custom_id": "k__v3", "response": None,
               "error": {"code": "unexpected_failure", "message": "Something else went wrong."}}, "review"),
    "l__v3": (ok_line("l__v3", 429, {"error": {"message": "You exceeded your project spend limit.", "type": "insufficient_quota",
                                               "param": None, "code": "project_spend_limit_exceeded"}}), "stop"),
    "m__v3": (ok_line("m__v3", 429, {"error": {"message": "You exceeded your current quota.", "type": "insufficient_quota",
                                               "param": None, "code": None}}), "stop"),
    "n__v3": (ok_line("n__v3", body=body_of([json.dumps({"vendor": None, "invoice_date": None, "currency": None,
                                                         "total": None})])), "review"),
}
EXPECTED_OK = {"vendor": "Acme Ltd", "invoice_date": "2026-03-03", "currency": "EUR", "total": 120.0}
DOCS = {f"doc-{i}": f"Vendor: V{i} | Date: 2026-01-0{i} | Currency: USD | Total: {i}0.00" for i in range(1, 8)}


def make_line(custom_id: str, document: str) -> dict:
    """Reference builder used by Exercise 3 (not the Exercise 1 answer: no strict schema here)."""
    return {"custom_id": custom_id, "method": "POST", "url": "/v1/responses",
            "body": {"model": MODEL, "instructions": INSTRUCTIONS, "input": f"<document>\n{document}\n</document>"}}


print("fixtures ready:", len(LINES), "sample result lines,", len(DOCS), "documents")

## Part A — Quiz

Eleven exam-style scenarios. Exactly one answer is defensible per question.

**Q1.** A team migrates an extraction service from GPT-5.5 to `gpt-6-luna`. Each request sends 2,500 tokens of rules in `instructions`, then a different document in the user message, in the default implicit caching mode. After the migration, input cost went **up**: `cache_write_tokens` is high on every request and `cached_tokens` is near 0. What is the best fix?

- A) Set `prompt_cache_retention="24h"` so cache entries live longer
- B) Send a unique `prompt_cache_key` per request so each request gets its own cache entry
- C) Move the rules into an `input_text` block of a `developer` message with an explicit `prompt_cache_breakpoint`, and use `prompt_cache_options={"mode": "explicit"}` so the changing documents are not written
- D) Shorten the rules below 1,024 tokens so nothing is ever written

**Q2.** A customer-facing assistant sends every request with `service_tier="fast"`. During a launch, traffic tripled within ten minutes. Afterwards, finance sees less Fast-mode spend than the request logs suggest, and p95 latency rose during the launch. What is the most likely explanation, and the fix?

- A) Some requests were downgraded to standard speed and billed at standard rates because traffic ramped too fast (the responses report `service_tier: "default"`). Ramp gradually, about 50% every 15 minutes once above 1M TPM, and log the **served** tier
- B) Fast mode silently moved the overflow to Flex, which is slower
- C) Fast mode has its own rate-limit pool, which ran out, so the overflow was queued as a batch
- D) The SDK retried the slow requests on Standard automatically

**Q3.** A Build-tier project on `gpt-6-luna` (5,000 RPM, 2,000,000 TPM) sends requests of about 1,000 input and 300 output tokens. The code sets `max_output_tokens=32000` "to be safe". The team hits `429` errors at about 60 requests per minute. What is the best first change?

- A) Upgrade to the Grow tier
- B) Move the interactive traffic to the Batch API
- C) Increase the number of retries with exponential backoff
- D) Size `max_output_tokens` per task from observed `usage` (a few thousand tokens leaves reasoning headroom), because the rate-limit estimate is based on that cap

**Q4.** A job must process 120,000 documents with the Batch API. A router sends 15% of them to `gpt-6.1-sol` and the rest to `gpt-6-luna`. Each file stays well under 200 MB. What is the minimum number of batches?

- A) 1, with `model` set per line
- B) 4: Luna's 102,000 lines need three files (at most 50,000 lines each) and Sol's 18,000 lines need one
- C) 3: the line limit is 50,000 per batch and 120,000 / 50,000 rounds up to 3
- D) 2: one per model

**Q5.** A Flex job wraps calls in a helper that, on any `429`, backs off and then falls back to Standard. One night every request returns `429` with `error.code == "credit_balance_exhausted"`, and the helper keeps cycling. What is wrong?

- A) Nothing; that is the documented Flex pattern for every `429`
- B) Flex `429`s are billed, so the fallback doubles the bill
- C) Billing and quota `429`s must stop and alert a human; Standard fails the same way. Only capacity `429`s on Flex ("Resource Unavailable") justify backoff or a fallback
- D) The helper should fall back to Fast mode, which has its own credit pool

**Q6.** Every call in a service is wrapped in an application retry loop with 5 attempts. During a `503 server_is_overloaded` incident, request volume to OpenAI was about 15× normal per user action. What change addresses the root cause?

- A) Set `max_retries=0` on the client used inside the loop (or remove the loop and configure the SDK), honor `Retry-After`, and bound the total retry time
- B) Raise the loop to 10 attempts so more calls eventually succeed
- C) Retry `503`s immediately without delay, since overload is brief
- D) Catch the `503` and send the request to a different project's key

**Q7.** To save money, a team switches a nightly job of 30,000 independent requests from synchronous calls to `responses.create(..., background=True)`, polling each response. What happens to the bill?

- A) It drops by 50%, because background responses use batch pricing
- B) Nothing changes: background mode is an asynchronous call pattern at the normal price of its tier. For the 50% discount, use the Batch API (or Flex if they need per-request synchronous calls)
- C) The requests fail, because background mode only works inside a batch file
- D) It drops by 50% only when `store=False`

**Q8.** A batch completed on 1 March. A monthly audit job tries to download its `output_file_id` on 3 April and gets a not-found error. What happened, and what is the design fix?

- A) Output files exist only for 24 hours after completion; poll more often
- B) Batch output files are deleted 30 days after completion by default (configurable from 1 hour to 30 days with `output_expires_after`). Copy results into your own store as soon as the batch finishes, keyed by `custom_id`
- C) The audit job needs an Admin API key to read batch files
- D) Batch files can only be read with the API key that created them

**Q9.** After fixing a prompt bug, a team re-ran 3,000 failed documents with prompt v4, reusing the same `custom_id`s as the v3 run (`doc-123`). Their results table now mixes v3 and v4 outputs, and they cannot tell which is which. What should the `custom_id` design have been?

- A) A fresh UUID per line, so IDs never collide
- B) The line number in the input file, so results can be sorted back into order
- C) A deterministic ID derived from the business key **and** the prompt version (for example `doc-123__invoice-v4`), recorded in a manifest with the batch ID and model
- D) A timestamp plus the document ID

**Q10.** A Luna → Sol cascade (escalate when Luna's output fails validation) cut extraction cost about 12×. A month later, an audit finds accuracy fell from 99% (Sol only) to 95%. The escalation rate is a steady 3%. What is the most likely cause, and the fix?

- A) Sol's accuracy dropped after a model update; switch everything to `gpt-6-astra`
- B) The Batch API lowers output quality; move the cascade to synchronous calls
- C) Prompt caching changed the outputs; disable caching
- D) Luna produces **valid but wrong** outputs that pass the validator and never escalate. Strengthen the validator or add an evidence signal, and gate the routing change on the chapter 06 eval with per-segment results before shipping

**Q11.** A team moves its nightly ETL into a new project whose allowed service tiers are `default` and `flex`, and the project's own tier setting is still `fast` from an earlier experiment. The ETL code never sets `service_tier`. Every request now fails with `400` `invalid_request_error`, `param: "service_tier"`. The on-call engineer proposes wrapping the calls in exponential backoff. What is the right response?

- A) Do not retry: the request resolves to the project's Fast tier (evaluated as `priority`), which the project disallows. Send `service_tier="flex"` or `"default"` explicitly, or change the project's tier setting, and keep Fast disallowed for bulk work
- B) Backoff is correct: a `400` on `service_tier` means Fast capacity is temporarily unavailable
- C) Add `priority` to the allowed tiers, because omitted `service_tier` always means Standard
- D) Move the ETL to the organization's default project, which has no restrictions

In [ ]:
# Fill in one letter per question ("A"-"D"). Leave None to skip.
answers = {
    "q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
    "q6": None, "q7": None, "q8": None, "q9": None, "q10": None,
    "q11": None,
}

In [ ]:
import hashlib

_KEY = {
    "q1": "227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e",
    "q2": "3d9c97ee45947882f5eee5cf878c4f33a0a270d8f2a2e1fc259d66cdc95c7b45",
    "q3": "14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6",
    "q4": "61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa",
    "q5": "e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f",
    "q6": "ac78c6455837b3b074080e7912c1bdab72fbbfdcaeefd9269fe293a50c4a9522",
    "q7": "16dd052a869d89a87df9ae09526a5477a3e6870575379b0c9d2ed35de3ab21c7",
    "q8": "39f95f59afaa9522616af26bd34df1e0184030ae8f738e319fc9a1f15b075834",
    "q9": "e19608e84d37f3c94b5f88738c75a8c96772a5e61784e3e24b5512fa65ff63c7",
    "q10": "c72adbd855b96e33ffae9308bfc1d5cdfa7fb315301530c3b1f890435529ec04",
    "q11": "acb9b8f6dd2a3e57b9476b90caf32ab8ad47a11c4aea20d95df7aaad0d81238c",
}


def grade_quiz(answers: dict) -> float:
    correct = 0
    for qid, digest in _KEY.items():
        letter = str(answers.get(qid) or "").strip().upper()
        ok = bool(letter) and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == digest
        correct += ok
        print(f"{qid:>3}: {'✓' if ok else '✗'}  (your answer: {letter or '-'})")
    score = correct / len(_KEY)
    print(f"Quiz score: {correct}/{len(_KEY)} = {score:.0%}")
    return score


quiz_score = grade_quiz(answers)

## Part B — Coding exercises

All exercises are offline. Implement the function in the stub cell, then run the checker cell below it. Exercises 2 and 3 fit together: `triage_line` produces the buckets that `build_retry_lines` consumes.

### Exercise 1 — Build a valid `/v1/responses` batch line

Write `build_batch_line(doc_id, document, prompt_version="invoice-v3", model=MODEL)` that returns **one JSONL line** (as a dict) for an invoice extraction batch:

- `custom_id`: `f"{doc_id}__{prompt_version}"`. `__` is the reserved separator that later code splits on, so if `doc_id` **or** `prompt_version` contains `"__"`, raise `ValueError` (otherwise `region__123` would come back as `region`, and `("a__b", "c")` would collide with `("a", "b__c")`). If the result does not match `CUSTOM_ID_RE` (`[A-Za-z0-9_-]`, 1–64 chars), raise `ValueError`. It must be deterministic (no timestamps, no UUIDs).
- `method` `"POST"`, `url` `"/v1/responses"`, and a `body` that contains **only Responses API parameters** (no `messages`, no `max_tokens`, no `stream`).
- `body["model"]` is `model`; `body["instructions"]` is exactly `INSTRUCTIONS` (identical on every line, so it is a cacheable prefix); `body["input"]` wraps the document in `<document>` … `</document>` tags; the document never appears in `instructions`.
- Structured output: `body["text"]["format"]` is a strict `json_schema` format named `"invoice"` with `INVOICE_SCHEMA`.
- `body["reasoning"]` is `{"effort": "low"}` and `body["max_output_tokens"]` is a deliberate cap between 500 and 4,000.

In [ ]:
def build_batch_line(doc_id: str, document: str, prompt_version: str = "invoice-v3", model: str = MODEL) -> dict:
    # TODO: return the JSONL line described above
    raise NotImplementedError

In [ ]:
def check_ex1():
    try:
        doc = "Vendor: Acme Ltd | Date: 2026-03-03 | Currency: EUR | Total: 120.00"
        line = build_batch_line("inv-000123", doc)
        assert isinstance(line, dict), "return a dict"
        assert set(line) == {"custom_id", "method", "url", "body"}, f"top-level keys must be custom_id/method/url/body, got {sorted(line)}"
        assert line["custom_id"] == "inv-000123__invoice-v3", f"custom_id should be 'inv-000123__invoice-v3', got {line['custom_id']!r}"
        assert line == build_batch_line("inv-000123", doc), "the line must be deterministic (same input -> same line)"
        assert build_batch_line("inv-000123", doc, prompt_version="invoice-v4")["custom_id"] == "inv-000123__invoice-v4", \
            "honor the prompt_version argument in the custom_id"
        assert line["method"] == "POST" and line["url"] == "/v1/responses", "method POST, url /v1/responses"
        body = line["body"]
        unknown = set(body) - BODY_KEYS
        assert not unknown, f"not Responses API parameters: {sorted(unknown)} (Chat Completions habits?)"
        assert not body.get("stream"), "batch lines cannot stream"
        BODY_ADAPTER.validate_python(body)
        assert body.get("model") == MODEL, "use the model argument"
        assert build_batch_line("x", doc, model="gpt-6.1-sol")["body"]["model"] == "gpt-6.1-sol", "honor the model argument"
        assert body.get("instructions") == INSTRUCTIONS, "instructions must be exactly INSTRUCTIONS (stable prefix)"
        other = build_batch_line("inv-000124", "Vendor: Other | Date: 2026-01-01 | Currency: USD | Total: 1.00")
        assert other["body"]["instructions"] == body["instructions"], "instructions must be identical on every line"
        assert isinstance(body.get("input"), str) and f"<document>\n{doc}\n</document>" in body["input"], \
            "input must wrap the document as <document>\\n...\\n</document>"
        assert doc not in body["instructions"], "the document belongs in input, not instructions"
        fmt = (body.get("text") or {}).get("format") or {}
        assert fmt.get("type") == "json_schema" and fmt.get("strict") is True and fmt.get("name") == "invoice", \
            "text.format must be a strict json_schema named 'invoice'"
        assert fmt.get("schema") == INVOICE_SCHEMA, "use INVOICE_SCHEMA as the schema"
        assert body.get("reasoning") == {"effort": "low"}, "reasoning must be {'effort': 'low'}"
        cap = body.get("max_output_tokens")
        assert isinstance(cap, int) and 500 <= cap <= 4000, "set max_output_tokens to a deliberate cap between 500 and 4,000"
        for bad in ["jane.doe@example.com", "orders/2026/7", "x" * 70, "region__123"]:
            try:
                build_batch_line(bad, doc)
            except ValueError:
                continue
            raise AssertionError(f"doc_id {bad!r} should raise ValueError (pattern [A-Za-z0-9_-], max 64 chars, no '__' inside)")
        try:
            build_batch_line("inv-1", doc, prompt_version="invoice__v4")
            raise AssertionError("a prompt_version containing '__' should raise ValueError (reserved separator)")
        except ValueError:
            pass
        print("✓ Exercise 1 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 1: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 1: {e}")
    except Exception as e:
        print(f"✗ Exercise 1 raised {type(e).__name__}: {e}")
    return False


ex1_passed = check_ex1()

### Exercise 2 — Classify output and error lines

Write `triage_line(line)` for one line from a batch output **or** error file. Return `(bucket, invoice_dict_or_None)` where `bucket` is one of `"ok"`, `"resubmit"`, `"fix_first"`, `"review"`, `"stop"`. Apply the rows **in order**:

| Line | Bucket |
|---|---|
| `error` object with `code == "batch_expired"` | `resubmit` |
| any other `error` object | `review` |
| non-200 `status_code` whose `response.body.error.code` is in `STOP_CODES`, or whose `error.type` is `"insufficient_quota"` (billing, spend, quota) | `stop` |
| `response.status_code` is `429` or `>= 500` | `resubmit` |
| any other non-200 `status_code` (for example `400`) | `fix_first` |
| `200`, but `body["status"] != "completed"` (for example `incomplete`) | `fix_first` |
| `200` and completed, but the joined output text fails `Invoice.model_validate_json`, or `vendor` or `total` is `None` (schema-valid, but not an invoice), or `currency` is not `None` and not three uppercase letters | `review` |
| otherwise | `ok`, with `Invoice(...).model_dump()` as the second element |

Join the text of **every** `output_text` part of every `message` item in `body["output"]`, skipping other item types (such as `reasoning`). Return `None` as the second element for every bucket except `ok`.

In [ ]:
def triage_line(line: dict) -> tuple[str, dict | None]:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2():
    try:
        for cid, (line, expected) in LINES.items():
            result = triage_line(line)
            assert isinstance(result, tuple) and len(result) == 2, "return a (bucket, invoice_or_None) tuple"
            bucket, invoice = result
            assert bucket == expected, f"{cid}: expected {expected!r}, got {bucket!r} (line: {json.dumps(line)[:160]}...)"
            if expected == "ok":
                want = {**EXPECTED_OK, "currency": None} if cid == "i__v3" else EXPECTED_OK
                assert invoice == want, f"{cid}: return the complete parsed invoice as a dict, expected {want}, got {invoice}"
            else:
                assert invoice is None, f"{cid}: return None as the second element for non-ok lines"
        print("✓ Exercise 2 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 2: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 2: {e}")
    except Exception as e:
        print(f"✗ Exercise 2 raised {type(e).__name__}: {e}")
    return False


ex2_passed = check_ex2()

### Exercise 3 — Build a resubmission file from the manifest

Write `build_retry_lines(manifest_ids, buckets, docs, builder)`:

- `manifest_ids`: every `custom_id` you submitted, in manifest order (for example `"doc-3__v3"`).
- `buckets`: `custom_id -> bucket` for every line that came back (output and error files), as returned by `triage_line`.
- `docs`: `doc_id -> source text`, where `doc_id` is the part of the `custom_id` before `"__"` (unambiguous, because Exercise 1 rejects `"__"` inside a part).
- `"stop"` lines (billing) are **not** resubmitted: a human fixes billing first.
- `builder(custom_id, document) -> dict` builds one line.

Return the list of lines to resubmit **as-is**: every manifest ID whose bucket is `"resubmit"`, **plus** every manifest ID that never came back at all (missing from `buckets`). Rebuild each line from `docs` with the **same** `custom_id`, never from the output file. Keep manifest order, no duplicates, and ignore IDs in `buckets` that are not in the manifest. `fix_first`, `review` and `ok` lines are not resubmitted.

In [ ]:
def build_retry_lines(manifest_ids: list[str], buckets: dict[str, str], docs: dict[str, str],
                      builder: Callable[[str, str], dict]) -> list[dict]:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3():
    try:
        manifest_ids = [f"doc-{i}__v3" for i in range(1, 8)] + ["doc-3__v3"]      # a duplicate slipped into the manifest
        buckets = {"doc-1__v3": "ok", "doc-2__v3": "resubmit", "doc-3__v3": "resubmit", "doc-4__v3": "fix_first",
                   "doc-5__v3": "review", "doc-7__v3": "resubmit", "stray__v3": "resubmit"}   # doc-6 never came back
        lines = build_retry_lines(manifest_ids, buckets, DOCS, make_line)
        assert isinstance(lines, list), "return a list of line dicts"
        ids = [l["custom_id"] for l in lines]
        assert "doc-6__v3" in ids, "a manifest ID that never came back (doc-6) must be resubmitted too"
        assert "stray__v3" not in ids, "ignore IDs that are not in the manifest"
        assert not {"doc-1__v3", "doc-4__v3", "doc-5__v3"} & set(ids), "only 'resubmit' and missing IDs: not ok, fix_first or review"
        assert ids == ["doc-2__v3", "doc-3__v3", "doc-6__v3", "doc-7__v3"], f"expected manifest order without duplicates, got {ids}"
        for l in lines:
            assert DOCS[l["custom_id"].split("__")[0]] in l["body"]["input"], "rebuild each line from docs with builder()"
        assert build_retry_lines(manifest_ids, {cid: "ok" for cid in manifest_ids}, DOCS, make_line) == [], \
            "nothing to resubmit when everything is ok"
        assert build_retry_lines(["doc-1__v3"], {"doc-1__v3": "stop"}, DOCS, make_line) == [], \
            "'stop' (billing) lines are not resubmitted: a human fixes billing first"
        print("✓ Exercise 3 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 3: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 3: {e}")
    except Exception as e:
        print(f"✗ Exercise 3 raised {type(e).__name__}: {e}")
    return False


ex3_passed = check_ex3()

### Exercise 4 — Cost of one request with cache reads, cache writes and tiers

Write `request_cost(model, usage, tier="default")` that returns the dollar cost of one request from a real `ResponseUsage` object, using the `PRICES` and `TIER_MULTIPLIER` fixtures:

- `usage.input_tokens` **includes** `usage.input_tokens_details.cached_tokens` and `usage.input_tokens_details.cache_write_tokens`. Charge the plain remainder at `input`, cached tokens at `cached`, written tokens at `write`.
- Charge `usage.output_tokens` (which includes reasoning tokens) at `output`.
- Prices are per **1M** tokens; multiply the total by `TIER_MULTIPLIER[tier]`. Ignore long context and regions.

In [ ]:
def request_cost(model: str, usage: ResponseUsage, tier: str = "default") -> float:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4():
    try:
        def close(a, b):
            return abs(a - b) < 1e-12

        steady = usage_of(2_000, 400, cached=1_200)
        got = request_cost("gpt-6-luna", steady)
        assert isinstance(got, float), "return a float (dollars)"
        assert not close(got, 0.000412), "cached tokens are part of input_tokens: don't charge them twice"
        assert close(got, 0.000292), f"gpt-6-luna steady state should cost $0.000292, got {got:.9f}"
        sol = request_cost("gpt-6.1-sol", steady)
        assert close(sol, 0.00572), f"gpt-6.1-sol should cost $0.00572 (cached reads are $0.10/1M there), got {sol:.9f}"
        written = request_cost("gpt-6-luna", usage_of(2_000, 400, written=1_200))
        assert close(written, 0.00043), f"1,200 written tokens are charged at the write price: expected $0.00043, got {written:.9f}"
        assert close(request_cost("gpt-6-astra", steady, "batch"), 0.0146), "batch halves the price"
        assert close(request_cost("gpt-6-luna", steady, "fast"), 0.000584), "fast doubles the price"
        reasoning_heavy = request_cost("gpt-6.1-sol", usage_of(300, 2_100, reasoning=2_000))
        assert close(reasoning_heavy, 0.0216), "reasoning tokens are already inside output_tokens; don't add them again"
        print("✓ Exercise 4 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 4: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 4: {e}")
    except Exception as e:
        print(f"✗ Exercise 4 raised {type(e).__name__}: {e}")
    return False


ex4_passed = check_ex4()

### Exercise 5 — Latest safe submission time for a batch

Write `latest_submit_time(deadline, now, *, retry_path_hours, safety_buffer_hours=1.0, window_hours=24.0)`:

- Return `deadline − window − retry path − safety buffer` as a `datetime`.
- If that moment is already **before** `now`, return `None` (the plan is infeasible: change the retry path or the processing tier). Exactly `now` is still feasible.

In [ ]:
def latest_submit_time(deadline: datetime, now: datetime, *, retry_path_hours: float,
                       safety_buffer_hours: float = 1.0, window_hours: float = 24.0) -> datetime | None:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5():
    try:
        now = datetime(2026, 10, 3, 9, 0, tzinfo=timezone.utc)
        deadline = now + timedelta(hours=30)
        t = latest_submit_time(deadline, now, retry_path_hours=0, safety_buffer_hours=0)
        assert t == now + timedelta(hours=6), f"30 h deadline, 24 h window, no retry path, no buffer -> now + 6 h, got {t}"
        t = latest_submit_time(deadline, now, retry_path_hours=3)
        assert t == now + timedelta(hours=2), f"with a 3 h Flex/Standard retry path and the default 1 h buffer -> now + 2 h, got {t}"
        assert latest_submit_time(deadline, now, retry_path_hours=24) is None, \
            "a second 24 h batch as the retry path does not fit in 30 h: return None"
        assert latest_submit_time(deadline, now, retry_path_hours=5, safety_buffer_hours=1) == now, \
            "a plan that leaves exactly zero slack is still feasible (return now, not None)"
        t = latest_submit_time(now + timedelta(days=3), now, retry_path_hours=24, window_hours=24)
        assert t == now + timedelta(hours=23), f"72 h deadline with a second batch and 1 h buffer -> now + 23 h, got {t}"
        print("✓ Exercise 5 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 5: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 5: {e}")
    except Exception as e:
        print(f"✗ Exercise 5 raised {type(e).__name__}: {e}")
    return False


ex5_passed = check_ex5()

### Exercise 6 — Retry policy by status and `error.code`

Write `retry_policy(status, code=None, *, err_type=None, flex=False)` that returns one of `"backoff"`, `"reduce_rate"`, `"flex_fallback"`, `"stop"`, `"fix"`. Apply the rules **in this order**:

1. `code` in `STOP_CODES` or `err_type == "insufficient_quota"` → `"stop"` (billing, spend, quota: even on Flex).
2. `status` `401` or `403` → `"stop"` (auth, region, or a monitoring stop such as `misalignment_policy_violation`).
3. `status` `429` with `code == "slow_down"` → `"reduce_rate"`.
4. Any other `429`: `"flex_fallback"` if `flex` else `"backoff"`.
5. `status is None` (connection error or timeout), `408`, `409`, or `>= 500` → `"backoff"`.
6. Anything else (for example `400` or `404`) → `"fix"`.

In [ ]:
def retry_policy(status: int | None, code: str | None = None, *, err_type: str | None = None, flex: bool = False) -> str:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6():
    try:
        CASES = [
            ((429, None), {}, "backoff"), ((429, "slow_down"), {}, "reduce_rate"), ((429, None), {"flex": True}, "flex_fallback"),
            ((429, "slow_down"), {"flex": True}, "reduce_rate"),
            ((429, "credit_balance_exhausted"), {"flex": True}, "stop"), ((429, "project_spend_limit_exceeded"), {}, "stop"),
            ((429, None), {"err_type": "insufficient_quota"}, "stop"), ((503, "server_is_overloaded"), {}, "backoff"),
            ((500, None), {}, "backoff"), ((None, None), {}, "backoff"), ((408, None), {}, "backoff"), ((409, None), {}, "backoff"),
            ((400, None), {}, "fix"), ((404, None), {}, "fix"), ((401, "invalid_api_key"), {}, "stop"),
            ((403, "misalignment_policy_violation"), {}, "stop"),
        ]
        for (status, code_), kwargs, expected in CASES:
            got = retry_policy(status, code_, **kwargs)
            assert got == expected, f"retry_policy({status}, {code_!r}, **{kwargs}) should be {expected!r}, got {got!r}"
        print("✓ Exercise 6 passed")
        return True
    except NotImplementedError:
        print("✗ Exercise 6: not implemented yet")
    except AssertionError as e:
        print(f"✗ Exercise 6: {e}")
    except Exception as e:
        print(f"✗ Exercise 6 raised {type(e).__name__}: {e}")
    return False


ex6_passed = check_ex6()

## Part C — Architecture scenario

A logistics company runs three AI workloads on the OpenAI platform, all currently synchronous Standard calls on `gpt-6.1-sol` from one project:

1. **Nightly risk scoring** of ~250,000 shipment records (each ~1,800 input tokens, of which ~1,400 are identical scoring rules and examples; ~300 output tokens). Scores must be in the warehouse by **06:00** for the operations dashboard. Records are final at **22:00** the previous evening.
2. **Same-day exception report**: ~3,000 delayed shipments are summarized for customers and must be emailed by **17:00**. Shipments are flagged throughout the day until 13:00.
3. **Dispatcher assistant**: an interactive chat for ~400 dispatchers, peaking at ~150 requests per minute, where answers must start streaming in under 2 seconds.

Last month: a $62,000 bill, frequent `429` errors in the dispatcher assistant during the nightly run, one night where an overnight retry loop ran for six hours against `project_spend_limit_exceeded`, and two mornings with missing scores after a "re-run everything" recovery.

Design the target architecture. Cover: the processing option and model for each workload (and why), the batch pipeline design (`custom_id`s, manifest, chunking, failure handling, retention), the deadline math for workload 1, caching layout, rate-limit and project separation, retry and error policy, and the cost model you would present (including how you would justify any model change). About 12–18 bullet points.

*Write your answer here (double-click to edit).*

<details><summary>Rubric: what a strong answer contains</summary>

- **Workload 1 → Batch API** (50% off, separate rate-limit pool, so it stops competing with the dispatcher assistant). Records are final at 22:00 and needed by 06:00: only **8 hours**, so a 24-hour completion window **cannot be guaranteed**. A strong answer spots this and proposes options: move the deadline, start scoring earlier with rolling submissions as records finalize, or run the tail through **Flex** with a Standard fallback (synchronous at batch prices) and treat Batch as the bulk path for whatever can be submitted earlier. It does not claim "most batches finish in an hour" as a plan.
- **Batch pipeline:** JSONL files of at most 50,000 lines (≥ 5 files for 250,000), one model per file; deterministic `custom_id` = shipment ID + prompt version; a manifest per batch (batch ID, file ID, model, prompt version, submitted IDs); join by `custom_id`, never by order; `missing = submitted − returned` check.
- **Failure handling:** classify output and error lines; resubmit only `batch_expired` / `429` / `5xx` lines (as-is), fix invalid `4xx` and `incomplete` lines first, route validation failures to retry-with-feedback or review. Never "re-run everything" (it caused the missing mornings and doubles cost). Webhooks (`batch.completed`, `batch.failed`, `batch.expired`) instead of polling.
- **Retention:** copy results to the warehouse on completion; output files are deleted after 30 days by default.
- **Workload 2 → Flex** with backoff and a Standard fallback when the 17:00 deadline gets close (or Batch only if flagged items can be submitted ≥ 24 h ahead, which they cannot). Latest-submit math with the retry path.
- **Workload 3 → Standard with streaming** on the cheapest model that passes its eval (consider `gpt-6-luna` or a Luna → Sol cascade for simple lookups); **Fast mode** only if latency is a measured product requirement, ramped gradually, with the served `service_tier` logged.
- **Caching:** the 1,400-token shared rules go first, identical on every request, in a `developer` `input_text` block with an explicit `prompt_cache_breakpoint`; explicit-only mode so per-record content is not written; per-tenant or per-workload `prompt_cache_key`; no timestamps in the prefix. Model batch costs **without** guaranteed cache hits.
- **Rate limits and projects:** separate projects for batch/offline work and the interactive assistant (and staging vs production), each with its own spend limit and alerts; right-size `max_output_tokens` because it drives the TPM estimate; remember cached tokens still count toward TPM; meter batch submissions against the queued-token limit; ramp new traffic gradually (`slow_down`).
- **Retry policy:** classify by status and `error.code`: back off on rate limit and overload (honor `Retry-After`), reduce rate on `slow_down`, **stop and alert** on spend, quota and credit codes (the six-hour loop), fix `400`s. `max_retries=0` on any client wrapped in an application retry loop.
- **Cost model:** per request from `usage` (input, cached, written, output including reasoning) × price × tier; per workload with retries, escalations and review; compare Sol vs Luna vs a cascade on the chapter 06 eval with per-segment results; report **cost per successful task**; reconcile daily with the Costs API grouped by project and line item; date the price table.
- **Rough numbers** (any reasonable estimate): workload 1 on Sol Standard ≈ 250,000 × (1,800 × $2 + 300 × $10) / 1M ≈ $1,650 per night; Batch halves it; caching the prefix and/or moving to Luna (if the eval passes) cuts it by a further large factor.
</details>

## Scorecard

In [ ]:
results = {name: globals().get(name, False) for name in
           ["ex1_passed", "ex2_passed", "ex3_passed", "ex4_passed", "ex5_passed", "ex6_passed"]}
exercises = sum(bool(v) for v in results.values()) / len(results)
quiz = globals().get("quiz_score", 0.0)
overall = 0.5 * quiz + 0.5 * exercises
print(f"Quiz:      {quiz:.0%}")
print(f"Exercises: {sum(bool(v) for v in results.values())}/{len(results)} passed ({exercises:.0%})")
print(f"Overall:   {overall:.0%}  ->  {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%, same bar as the Claude path)")

---

## Solutions (spoilers)

Try everything first. Each block below is collapsed.

<details><summary>Q1 answer</summary>

**C.** In implicit mode the breakpoint sits at the end of the latest user message, so each request writes rules + document (1.25×) and nothing matches the next request. An explicit breakpoint after the rules turns later requests into 0.1× reads, and explicit-only mode stops writing the documents. `instructions` cannot hold a breakpoint. A: `prompt_cache_retention` is for models before GPT-5.6; GPT-6 uses `prompt_cache_options.ttl`. B: a per-request key separates accounting and makes sharing impossible. D removes the write charge but also gives up the 90% read discount, so it is not the best fix.
</details>

<details><summary>Q2 answer</summary>

**A.** The Fast mode guide: if traffic ramps too fast, some requests are served at standard speed, billed at standard rates, and report `service_tier: "default"`. Its rule of thumb is to grow by no more than 50% every 15 minutes above 1M TPM. B and C invent behavior: Fast shares the Standard rate limit for the model. D: the SDK retries errors, it does not change tiers.
</details>

<details><summary>Q3 answer</summary>

**D.** The rate-limits guide says the rate-limit charge is the larger of `max_tokens` and the estimated token count of the request (the guide's wording; `max_output_tokens` is the Responses API cap, and the estimator is unpublished, so treat this as a planning approximation). 2,000,000 / 32,000 ≈ 62 requests per minute, which matches the symptom. A pays for a problem the code created. B breaks an interactive product. C adds traffic during the incident and fixes nothing.
</details>

<details><summary>Q4 answer</summary>

**B.** Each input file holds requests for a single model, and each batch holds at most 50,000 requests. 102,000 Luna lines → 3 files; 18,000 Sol lines → 1 file. A violates the one-model rule. C ignores the model split. D ignores the 50,000-line limit.
</details>

<details><summary>Q5 answer</summary>

**C.** OpenAI's error guide: retrying billing, spend or quota errors won't restore access; inspect `error.code` (the type can still be `insufficient_quota`). A misreads the Flex guidance, which is about capacity. B is false: Flex's Resource Unavailable `429` is not charged. D: Fast draws on the same credits.
</details>

<details><summary>Q6 answer</summary>

**A.** The Python SDK already retries `5xx`, `429`, `408`, `409` and connection errors twice, so 5 application attempts × 3 HTTP requests = 15. OpenAI's rate-limit guide says to disable SDK retries or account for them so nested loops don't multiply requests. B and C make the incident worse. D bypasses limits someone set on purpose.
</details>

<details><summary>Q7 answer</summary>

**B.** Background mode avoids client timeouts on long jobs; it is not a discount. Batch (50% off, within 24 hours, separate limits) is the right tool for a nightly bulk job; Flex is the synchronous alternative at batch prices. C and D invent constraints.
</details>

<details><summary>Q8 answer</summary>

**B.** The Batch guide states that the output file is deleted 30 days after the batch completes. The Batches API reference adds `output_expires_after` on `batches.create`: 3,600 to 2,592,000 seconds, counted from the file's `created_at`. Treat OpenAI's file as a delivery mechanism, not storage. A, C and D are invented constraints.
</details>

<details><summary>Q9 answer</summary>

**C.** IDs should be unique, derived from the business key, versioned with the prompt, and deterministic, so you can join results back, diff submitted against returned, and never confuse runs. A and D are not deterministic (you cannot rebuild or diff them) and A loses the business key. B breaks because output order may not match input order.
</details>

<details><summary>Q10 answer</summary>

**D.** A cascade only catches errors its validator can see. The README's exam trap: if Luna's wrong values pass validation, the cascade never escalates them and accuracy quietly drops. A, B and C blame components that do not change answers in the described way: caching and Batch do not alter model outputs, and nothing points to Sol.
</details>

<details><summary>Q11 answer</summary>

**A.** The error-codes guide: "Invalid service_tier argument" means the requested **or resolved** tier is not allowed for the project; requests that omit `service_tier` or send `auto` use the project's configured tier and can fail this way, and `fast` is evaluated as `priority`. It is a configuration error, so retrying the same request fails the same way. B confuses it with Flex's capacity `429`. C is wrong twice: omitted means the project tier (here Fast), and allowing it would bill ETL at 2×. D removes a guardrail instead of fixing the configuration.
</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
def build_batch_line(doc_id: str, document: str, prompt_version: str = "invoice-v3", model: str = MODEL) -> dict:
    if "__" in doc_id or "__" in prompt_version:
        raise ValueError("'__' is the reserved custom_id separator")
    custom_id = f"{doc_id}__{prompt_version}"
    if not CUSTOM_ID_RE.fullmatch(custom_id):
        raise ValueError(f"invalid custom_id {custom_id!r}")
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": model,
            "instructions": INSTRUCTIONS,
            "input": f"<document>\n{document}\n</document>",
            "reasoning": {"effort": "low"},
            "max_output_tokens": 2000,
            "text": {"format": {"type": "json_schema", "name": "invoice", "schema": INVOICE_SCHEMA, "strict": True}},
        },
    }
```
</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
def triage_line(line: dict) -> tuple[str, dict | None]:
    if line.get("error"):
        return ("resubmit" if line["error"].get("code") == "batch_expired" else "review"), None
    response = line.get("response") or {}
    code = response.get("status_code")
    if code != 200:
        err = (response.get("body") or {}).get("error") or {}
        if err.get("code") in STOP_CODES or err.get("type") == "insufficient_quota":
            return "stop", None
        return ("resubmit" if code == 429 or (code or 0) >= 500 else "fix_first"), None
    body = response.get("body") or {}
    if body.get("status") != "completed":
        return "fix_first", None
    text = "".join(part.get("text", "") for item in body.get("output", []) if item.get("type") == "message"
                   for part in item.get("content", []) if part.get("type") == "output_text")
    try:
        invoice = Invoice.model_validate_json(text)
    except ValidationError:
        return "review", None
    if invoice.vendor is None or invoice.total is None:
        return "review", None
    if invoice.currency is not None and not re.fullmatch(r"[A-Z]{3}", invoice.currency):
        return "review", None
    return "ok", invoice.model_dump()
```
</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
def build_retry_lines(manifest_ids: list[str], buckets: dict[str, str], docs: dict[str, str],
                      builder: Callable[[str, str], dict]) -> list[dict]:
    lines, seen = [], set()
    for cid in manifest_ids:
        if cid in seen:
            continue
        seen.add(cid)
        if cid not in buckets or buckets[cid] == "resubmit":
            lines.append(builder(cid, docs[cid.split("__")[0]]))
    return lines
```
</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
def request_cost(model: str, usage: ResponseUsage, tier: str = "default") -> float:
    p, d = PRICES[model], usage.input_tokens_details
    plain = usage.input_tokens - d.cached_tokens - d.cache_write_tokens
    dollars = (plain * p["input"] + d.cached_tokens * p["cached"] + d.cache_write_tokens * p["write"]
               + usage.output_tokens * p["output"]) / 1_000_000
    return dollars * TIER_MULTIPLIER[tier]
```
</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
def latest_submit_time(deadline: datetime, now: datetime, *, retry_path_hours: float,
                       safety_buffer_hours: float = 1.0, window_hours: float = 24.0) -> datetime | None:
    latest = deadline - timedelta(hours=window_hours + retry_path_hours + safety_buffer_hours)
    return None if latest < now else latest
```
</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
def retry_policy(status: int | None, code: str | None = None, *, err_type: str | None = None, flex: bool = False) -> str:
    if code in STOP_CODES or err_type == "insufficient_quota":
        return "stop"
    if status in (401, 403):
        return "stop"
    if status == 429:
        if code == "slow_down":
            return "reduce_rate"
        return "flex_fallback" if flex else "backoff"
    if status is None or status in (408, 409) or status >= 500:
        return "backoff"
    return "fix"
```
</details>

## Next chapter

Next: [Chapter 08 — Task Decomposition and Orchestration](../08-task-decomposition-and-orchestration/README.md).